# OCSA Deepfake Detection - Colab Baseline

Run this notebook in Google Colab to install the project, verify ResNet18, run a dummy train/evaluate smoke test, and save results. FaceForensics++ is opt-in and skipped by default.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

PROJECT_ROOT = Path('/content/OCSA-Deepfake-Detection')
REPOSITORY_URL = 'https://github.com/geethikagattu/OCSA-Deepfake-Detection.git'
if not (PROJECT_ROOT / 'requirements.txt').exists():
    subprocess.run(['git', 'clone', REPOSITORY_URL, str(PROJECT_ROOT)], check=True)
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT)) if str(PROJECT_ROOT) not in sys.path else None
print('Project root:', PROJECT_ROOT)

In [ ]:
%pip install -q -r requirements.txt

In [ ]:
import json
import numpy as np
import torch
from PIL import Image
from src.detector.dataset import create_dataloader
from src.detector.evaluate import evaluate
from src.detector.model import DeepfakeDetector
from src.detector.train import train_one_epoch

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('Device:', device)
model = DeepfakeDetector(pretrained=True).to(device)
with torch.no_grad():
    output = model(torch.randn(4, 3, 224, 224, device=device))
assert output.shape == (4,)
print('Baseline model smoke test: PASSED')

## Dummy-data pipeline smoke test

Random images are used only to verify data loading, training, and evaluation.

In [ ]:
dummy_root = Path('/content/test_dataset')
rng = np.random.default_rng(42)
for split in ['train', 'val']:
    for label in ['real', 'fake']:
        folder = dummy_root / split / label
        folder.mkdir(parents=True, exist_ok=True)
        for old_file in folder.glob('*.jpg'):
            old_file.unlink()
        for index in range(4):
            pixels = rng.integers(0, 256, size=(224, 224, 3), dtype=np.uint8)
            Image.fromarray(pixels).save(folder / f'{index:04d}.jpg')
train_dataset, train_loader = create_dataloader(dummy_root / 'train', batch_size=4, shuffle=True, num_workers=0)
val_dataset, val_loader = create_dataloader(dummy_root / 'val', batch_size=4, shuffle=False, num_workers=0)
criterion = torch.nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
loss = train_one_epoch(model, train_loader, optimizer, criterion, device)
metrics = evaluate(model, val_loader, device)
print('Classes:', train_dataset.classes)
print('Train images:', len(train_dataset))
print('Validation images:', len(val_dataset))
print('Dummy loss:', loss)
print('Dummy metrics:', metrics)
print('Dummy pipeline: PASSED')

In [ ]:
results_root = PROJECT_ROOT / 'experiments' / 'results'
results_root.mkdir(parents=True, exist_ok=True)
with open(results_root / 'dummy_smoke_test.json', 'w') as file:
    json.dump({'loss': loss, **metrics}, file, indent=2)
torch.save(model.state_dict(), results_root / 'dummy_resnet18.pt')
print('Saved results to:', results_root)

## Optional FaceForensics++ experiment

Enable this only after approved dataset access and agreement on splits and preprocessing. Keep downloaded videos outside Git.

In [ ]:
RUN_FFPP = False
if RUN_FFPP:
    raise NotImplementedError('Configure approved FaceForensics++ access and its official downloader before enabling this section.')
print('FF++ download skipped. The Colab baseline smoke test is complete.')